# SHAP-Based Patient-Level Explanation of Mortality Risk Predictions

In [53]:
import joblib
import pandas as pd
from sklearn.pipeline import Pipeline

# Load Saved Models and Data

In [55]:
# We load our best models
best_logistic_model = joblib.load("../machine learning modeling/best_logistic_model.pkl")
best_model_gb = joblib.load("../machine learning modeling/best_gradient_boosting_model.pkl")
X_train = joblib.load("../machine learning modeling/X_train.pkl")
X_test = joblib.load("../machine learning modeling/X_test.pkl")
y_train = joblib.load("../machine learning modeling/y_train.pkl")
y_test = joblib.load("../machine learning modeling/y_test.pkl")

In [56]:
print(type(best_logistic_model))
print(type(best_model_gb))

<class 'sklearn.pipeline.Pipeline'>
<class 'sklearn.pipeline.Pipeline'>


This shows the saved models are pipeline. So, we double check the pipeline steps.

In [58]:
if isinstance(best_logistic_model, Pipeline):
    print("Logistic Pipeline Steps:", best_logistic_model.named_steps.keys())

if isinstance(best_model_gb, Pipeline):
    print("Gradient Boosting Pipeline Steps:", best_model_gb.named_steps.keys())

Logistic Pipeline Steps: dict_keys(['preprocessor', 'classifier'])
Gradient Boosting Pipeline Steps: dict_keys(['preprocessor', 'classifier'])


We separate the preprocessing step from the estimator (classifier) so SHAP explainer sees the transformed features that the classifier uses.

In [60]:
def split_pipeline(model):
    if isinstance(model, Pipeline):
        preprocessor = Pipeline(model.steps[:-1]) # returns everything but the last item in the pipeline
        estimator = model.steps[-1][1]
        return preprocessor, estimator

    return None, model

log_preprocess, log_estimator = split_pipeline(best_logistic_model)
gb_preprocess, gb_estimator = split_pipeline(best_model_gb)

In [61]:
print(log_estimator)

LogisticRegression(C=0.03, class_weight='balanced', max_iter=1000, penalty='l2',
                   random_state=42)


In [62]:
print(gb_estimator)

HistGradientBoostingClassifier(class_weight='balanced', early_stopping=True,
                               l2_regularization=np.float64(1.025616274847307),
                               learning_rate=np.float64(0.012502377950801122),
                               max_features=0.6, max_iter=847,
                               max_leaf_nodes=77, n_iter_no_change=20,
                               random_state=42, validation_fraction=0.15)


# Prepare Data for SHAP

We start by defining functions that will help us transform our data

In [65]:
# After preprocessing, the transformed matrix may be a sparse matrix,
# which is not preferred for ploting, so we convert to dense matrix

def to_dense(X):
    if hasattr(X, "toarray"):        #sparse matrix has method "toarray" attributes
        return X.to_array()         # converts to dense arrays
    return X
        

def get_feature_names(preprocess, X_raw):
    """ We will try to extract the feature names and clean it, and if it fails we resort to default names"""
    try:
        names = preprocess.get_feature_names_out()
    except Exception:
        names = [f"feature_{i}" for i in range(preprocess.transform(X_raw).shape[1])]

    cleaned = []
    for name in names:
        cleaned_name = str(name)
        cleaned_name = cleaned_name.replace("num__", "")
        cleaned_name = cleaned_name.replace("cat__", "")
        cleaned_name = cleaned_name.replace("numeric__","")
        cleaned.append(cleaned_name)
    return cleaned

print(get_feature_names(gb_preprocess, X_train))

['age', 'creatinine_phosphokinase', 'ejection_fraction', 'platelets', 'serum_creatinine', 'serum_sodium', 'anaemia', 'diabetes', 'high_blood_pressure', 'sex', 'smoking']


We are now ready to transform our X_train and X_test into a form most compatible for SHAP

In [67]:
# We transform data for logistic regression
if log_preprocess is not None:     # means log_preprocess is in a pipeline
    X_train_log_array = to_dense(log_preprocess.transform(X_train))
    X_test_log_array = to_dense(log_preprocess.transform(X_test))
    log_feature_names = get_feature_names(log_preprocess, X_train)
else:    # we use the original data if not.
    X_train_log_array = X_train.values
    X_test_log_array = X_test.values
    log_feature_names = X_train.columns.tolist()

# We put the transformed data into a data frame.

X_train_log = pd.DataFrame(X_train_log_array,
                           columns = log_feature_names,
                           index = X_train.index)

X_test_log = pd.DataFrame(X_test_log_array, columns= log_feature_names, index = X_test.index)
    

In [68]:
X_train_log.head()

,age,creatinine_phosphokinase,ejection_fraction,platelets,serum_creatinine,serum_sodium,anaemia,diabetes,high_blood_pressure,sex,smoking
115,-0.269050,-0.200735,0.176528,-1.004722,-0.360437,0.559915,1.0,0.0,0.0,0.0,0.0
23,-0.706883,-0.534318,1.847425,1.051685,-0.544467,-0.345802,0.0,1.0,0.0,1.0,0.0
0,1.219579,-0.020580,-1.494369,0.013401,0.467698,-1.477949,0.0,0.0,1.0,1.0,0.0
247,0.256348,-0.455129,-1.076645,-0.178127,0.927773,-0.345802,0.0,0.0,0.0,1.0,0.0
194,-1.407414,-0.020580,-1.494369,-1.387778,0.191653,-0.345802,0.0,0.0,1.0,1.0,0.0


In [69]:
X_test_log.head()

,age,creatinine_phosphokinase,ejection_fraction,platelets,serum_creatinine,serum_sodium,anaemia,diabetes,high_blood_pressure,sex,smoking
274,-0.093917,-0.342285,-0.658921,-1.145848,-0.360437,0.107057,1.0,1.0,0.0,1.0,1.0
149,-0.093917,1.641397,-0.241196,-0.359574,-0.452452,-0.119373,0.0,0.0,1.0,1.0,0.0
120,-0.093917,0.132848,1.847425,-0.541022,0.099638,-0.345802,1.0,0.0,1.0,1.0,1.0
226,-0.269050,-0.540257,-1.076645,-0.752711,-0.084392,-1.025090,1.0,0.0,0.0,1.0,1.0
125,-1.582547,-0.242309,1.011976,-0.268851,-0.084392,-0.345802,1.0,0.0,0.0,0.0,0.0


In [70]:
print(X_train_log.shape)
print(X_test_log.shape)

(239, 11)
(60, 11)


Gradient boosting transformed data

In [72]:
if gb_preprocess is not None:
    X_train_gb_array = to_dense(gb_preprocess.transform(X_train))
    X_test_gb_array = to_dense(gb_preprocess.transform(X_test))
    gb_feature_names = get_feature_names(gb_preprocess, X_train)

else:
    X_train_gb_array = X_train.values
    X_test_gb_array = X_test.values
    gb_feature_names = X_train.columns.tolist()


X_train_gb = pd.DataFrame(X_train_gb_array, 
                          columns = gb_feature_names,
                          index= X_train.index)

X_test_gb = pd.DataFrame(X_test_gb_array,
                         columns = gb_feature_names,
                         index = X_test.index)



In [73]:
print(X_train_gb.shape)
print(X_test_gb.shape)

(239, 11)
(60, 11)


In [74]:
X_train_gb.head()

,age,creatinine_phosphokinase,ejection_fraction,platelets,serum_creatinine,serum_sodium,anaemia,diabetes,high_blood_pressure,sex,smoking
115,58.0,400.0,40.0,164000.0,1.0,139.0,1.0,0.0,0.0,0.0,0.0
23,53.0,63.0,60.0,368000.0,0.8,135.0,0.0,1.0,0.0,1.0,0.0
0,75.0,582.0,20.0,265000.0,1.9,130.0,0.0,0.0,1.0,1.0,0.0
247,64.0,143.0,25.0,246000.0,2.4,135.0,0.0,0.0,0.0,1.0,0.0
194,45.0,582.0,20.0,126000.0,1.6,135.0,0.0,0.0,1.0,1.0,0.0


In [91]:
X_test_gb.head()

,age,creatinine_phosphokinase,ejection_fraction,platelets,serum_creatinine,serum_sodium,anaemia,diabetes,high_blood_pressure,sex,smoking
274,60.0,257.0,30.0,150000.0,1.0,137.0,1.0,1.0,0.0,1.0,1.0
149,60.0,2261.0,35.0,228000.0,0.9,136.0,0.0,0.0,1.0,1.0,0.0
120,60.0,737.0,60.0,210000.0,1.5,135.0,1.0,0.0,1.0,1.0,1.0
226,58.0,57.0,25.0,189000.0,1.3,132.0,1.0,0.0,0.0,1.0,1.0
125,43.0,358.0,50.0,237000.0,1.3,135.0,1.0,0.0,0.0,0.0,0.0
